In [ ]:
# Example 1: FlaskInstrumentor class exists and exposes the expected API.
from opentelemetry.instrumentation.flask import FlaskInstrumentor

instrumentor = FlaskInstrumentor()

# Standard BaseInstrumentor methods
assert hasattr(instrumentor, 'instrument'), 'missing instrument()'
assert hasattr(instrumentor, 'uninstrument'), 'missing uninstrument()'
assert hasattr(instrumentor, 'instrumentation_dependencies'), 'missing instrumentation_dependencies()'
assert hasattr(FlaskInstrumentor, 'instrument_app'), 'missing instrument_app()'
assert hasattr(FlaskInstrumentor, 'uninstrument_app'), 'missing uninstrument_app()'

deps = instrumentor.instrumentation_dependencies()
print(f"instrumentation_dependencies = {list(deps)}")
assert any('flask' in d.lower() for d in deps), 'expected flask in dep list'


In [ ]:
# Example 2: instrument a Flask app and verify a span is recorded for a request.
# The tracer_provider is passed explicitly — trace.set_tracer_provider() only honors
# its FIRST call per process, so relying on the global provider is order-fragile.
from flask import Flask
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.flask import FlaskInstrumentor

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))

app = Flask(__name__)
FlaskInstrumentor().instrument_app(app, tracer_provider=provider)

@app.route('/hello')
def hello():
    return 'ok'

client = app.test_client()
resp = client.get('/hello')
assert resp.status_code == 200

spans = exporter.get_finished_spans()
print(f"finished spans: {len(spans)}")
assert len(spans) >= 1, 'expected at least one span'
FlaskInstrumentor().uninstrument_app(app)

In [ ]:
# Example 3: span attributes include HTTP method and status from the request.
from flask import Flask
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.flask import FlaskInstrumentor

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))

app = Flask(__name__)
FlaskInstrumentor().instrument_app(app, tracer_provider=provider)

@app.route('/echo')
def echo():
    return {'ok': True}, 201

app.test_client().get('/echo')
spans = exporter.get_finished_spans()
assert spans, 'expected at least one finished span'
attrs = spans[-1].attributes
print(dict(attrs))
method_keys = [k for k in attrs if 'method' in k.lower()]
assert method_keys, f"expected an http method attribute, got {list(attrs)}"
FlaskInstrumentor().uninstrument_app(app)

In [ ]:
# Example 4: excluded_urls option suppresses spans for matching paths.
from flask import Flask
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.flask import FlaskInstrumentor

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))

app = Flask(__name__)
FlaskInstrumentor().instrument_app(app, excluded_urls='healthz', tracer_provider=provider)

@app.route('/healthz')
def healthz():
    return 'ok'

@app.route('/work')
def work():
    return 'ok'

client = app.test_client()
client.get('/healthz')
client.get('/work')
spans = exporter.get_finished_spans()
assert spans, 'expected the /work request to produce a span'
paths = [s.attributes.get('http.target') or s.attributes.get('url.path') or s.name for s in spans]
print(paths)
assert not any('healthz' in str(p) for p in paths), 'healthz should be excluded'
FlaskInstrumentor().uninstrument_app(app)

In [ ]:
# Example 5: uninstrument cleanly removes the instrumentation; subsequent requests record no spans.
from flask import Flask
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.flask import FlaskInstrumentor

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))

app = Flask(__name__)
FlaskInstrumentor().instrument_app(app, tracer_provider=provider)

@app.route('/ping')
def ping():
    return 'pong'

app.test_client().get('/ping')
before = len(exporter.get_finished_spans())

FlaskInstrumentor().uninstrument_app(app)
exporter.clear()
app.test_client().get('/ping')
after = len(exporter.get_finished_spans())

print(f"spans before uninstrument={before}, after uninstrument={after}")
assert before >= 1 and after == 0, 'uninstrument_app should stop span emission'